# Smoke test Kaggle — avant toute vraie session

**Objectif : dépenser deux minutes de GPU pour lever quatre inconnues** qui, sinon, se
révéleraient au milieu d'une run longue.

1. `transformers` de l'image Kaggle connaît-il `qwen3_5` ? En dessous de 5.12.1, le modèle ne
   charge pas du tout.
2. Un checkpoint multimodal `qwen3_5` se charge-t-il bien en texte seul ?
3. `Qwen3.5-4B-Base` **suit-il le format de réponse** ? Il n'est pas instruct. S'il ne le suit
   pas, le juge a besoin d'une passe SFT avant le DPO — et le plan de la semaine change.
4. `DPOTrainer` accepte-t-il cette architecture ?

**Ce qu'on regarde n'est pas le score.** C'est le taux de sorties non parsables.

---

### Réglages Kaggle requis

- **Accelerator** : GPU T4 x2
- **Internet** : activé (les entrées « modèle » de Kaggle pointent vers Hugging Face, ce ne
  sont pas des copies hors ligne)
- **Secrets** : `GITHUB_PAT` (dépôt privé) et `HF_TOKEN` (évite le rate-limit)

## 1. Environnement

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

In [ ]:
import importlib.metadata as md_

for p in ["torch", "transformers", "datasets", "accelerate", "peft", "trl", "bitsandbytes"]:
    try:
        print(f"{p:<16}{md_.version(p)}")
    except Exception:
        print(f"{p:<16}ABSENT")

### Mise à jour de transformers

C'est la dépendance critique. Les deux backbones sont `model_type: qwen3_5`, inconnu avant
la 5.12.1 — sur une image plus ancienne, le chargement échoue sur une erreur de configuration
non reconnue.

⚠️ **Redémarrer le kernel après cette cellule** si transformers a réellement été mis à jour.

In [ ]:
!pip install -q -U "transformers>=5.12.1" trl peft bitsandbytes accelerate

## 2. Récupérer le code et les données

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_TOKEN")
pat = secrets.get_secret("GITHUB_PAT")

REPO = "afrique-safety-dpo_alignment"
if not os.path.exists(REPO):
    !git clone -q https://{pat}@github.com/zoom-BT/{REPO}.git
%cd {REPO}
!git log --oneline -1

In [ ]:
# Les données UbuntuGuard sont versionnées avec le code : rien à uploader.
!ls -la data/*.jsonl

from src.data import build_guardian_pairs, load_ubuntuguard_rows
from src.run_guardian_eval import load_eval_pairs
import yaml

config = yaml.safe_load(open("config.yaml"))
print("\naxe :", config["dpo"]["axis"])
print("tranche d'eval :", len(load_eval_pairs(config)), "exemples")

## 3. Inconnue n°1 — `transformers` connaît-il `qwen3_5` ?

Vérifié sans télécharger un seul octet de poids.

In [ ]:
from transformers.models.auto import modeling_auto as ma

mapping = ma.MODEL_FOR_CAUSAL_LM_MAPPING_NAMES
if "qwen3_5" in mapping:
    print("OK  qwen3_5 ->", mapping["qwen3_5"])
else:
    print("ECHEC : cette version de transformers ne connait pas qwen3_5.")
    print("        Relancer la cellule pip puis REDEMARRER LE KERNEL.")

In [ ]:
from transformers import AutoConfig

MODELES = {
    "cible":   "McGill-NLP/AfriqueQwen3.5-4B-50Langs",
    "controle": "Qwen/Qwen3.5-4B-Base",
}
for role, nom in MODELES.items():
    c = AutoConfig.from_pretrained(nom)
    sous = [k for k, v in vars(c).items() if hasattr(v, "model_type")]
    print(f"{role:<9} {nom}")
    print(f"          model_type={c.model_type}  architectures={getattr(c, 'architectures', None)}")
    print(f"          sous-configs={sous}   <- multimodal, on ne chargera que la partie texte")

## 4. Inconnue n°2 — le tokenizer et le template de chat

`Qwen3.5-4B-Base` n'est pas instruct et n'a probablement **pas** de template de chat. Notre
`render_prompt` retombe alors sur un rendu `role: content` en clair. On vérifie lequel des
deux chemins sera pris, avant de générer quoi que ce soit.

In [ ]:
from transformers import AutoTokenizer
from src.run_guardian_eval import render_prompt

tokenizers = {}
for role, nom in MODELES.items():
    tok = AutoTokenizer.from_pretrained(nom)
    tokenizers[role] = tok
    a_template = bool(getattr(tok, "chat_template", None))
    print(f"{role:<9} vocab={tok.vocab_size:>7}  template de chat : {'oui' if a_template else 'NON -> repli texte brut'}")

In [ ]:
paires = load_eval_pairs(config)[:20]
apercu = render_prompt(paires[0]["prompt"], tokenizers["controle"])
print(apercu[:600], "\n...\n")
print("longueur en tokens :", len(tokenizers["controle"](apercu)["input_ids"]))

## 5. Inconnue n°3 — le modèle suit-il le format de réponse ?

**C'est la cellule qui décide de la suite de la semaine.**

Le prompt demande `<answer>PASS</answer>`. Un modèle base peut très bien partir dans une
digression sans jamais produire ce bloc. Le script avertit au-delà de 20 % de sorties non
parsables.

In [ ]:
!python -m src.run_guardian_eval --model Qwen/Qwen3.5-4B-Base --limit 20 --batch-size 4

In [ ]:
!python -m src.run_guardian_eval --model McGill-NLP/AfriqueQwen3.5-4B-50Langs --limit 20 --batch-size 4

### Lire les sorties brutes

Les scores sur 20 exemples ne veulent rien dire. Les **sorties**, si : elles montrent si le
modèle a compris la tâche ou s'il s'est mis à bavarder.

In [ ]:
import glob, json

import pandas as pd

for chemin in sorted(glob.glob("results/guardian/records_*.jsonl")):
    lignes = [json.loads(l) for l in open(chemin, encoding="utf-8")]
    print("===", chemin, f"({len(lignes)} lignes)")
    df = pd.DataFrame([
        {
            "vrai": r["label"],
            "predit": r["predicted"],
            "strict": r["predicted_strict"],
            "sortie brute": r["completion"][:110].replace("\n", " "),
        }
        for r in lignes[:10]
    ])
    display(df)
    non_parsable = sum(r["predicted_strict"] == "UNKNOWN" for r in lignes) / len(lignes)
    print(f"non parsable (strict) : {non_parsable:.0%}\n")

## 6. Inconnue n°4 — `DPOTrainer` accepte-t-il cette architecture ?

Deux pas d'optimisation, sur le plus petit lot possible. On ne cherche pas à apprendre quoi
que ce soit : on cherche à savoir si ça **plante**.

In [ ]:
import torch
from trl import DPOConfig, DPOTrainer

from src.data import split_three_way
from src.train import build_peft_config, build_dpo_dataset_from_pairs, load_causal_lm

rows = load_ubuntuguard_rows(config["dpo"]["ubuntuguard_path"])
juge, _, _ = split_three_way(build_guardian_pairs(rows))
petit = build_dpo_dataset_from_pairs(juge[:8])

tok = tokenizers["controle"]
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

modele = load_causal_lm(MODELES["controle"], config["training"])

args = DPOConfig(
    output_dir="/kaggle/working/smoke",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,
    max_steps=2,
    learning_rate=5e-6,
    beta=0.1,
    max_length=config["training"]["max_seq_length"],
    bf16=True,
    report_to="none",
    logging_steps=1,
)
trainer = DPOTrainer(
    model=modele,
    ref_model=None,
    args=args,
    train_dataset=petit,
    peft_config=build_peft_config(config["training"]),
    processing_class=tok,
)
trainer.train()
print("\nOK — DPOTrainer accepte le checkpoint qwen3_5.")
print("VRAM max :", round(torch.cuda.max_memory_allocated() / 1e9, 2), "Go")

---

## 7. Que faire selon ce qu'on a vu

| Observation | Signification | Suite |
| :---- | :---- | :---- |
| Non parsable < 20 % sur les deux | les modèles suivent le format | **Plan inchangé** : DPO direct sur la tâche gardien |
| Non parsable élevé sur le contrôle seul | le modèle base ne suit pas les consignes | Ajouter une **passe SFT sur le format** avant le DPO du juge |
| Non parsable élevé sur les deux | le prompt ne passe pas tel quel | Envisager du **few-shot** dans le prompt gardien, ou un SFT sur les deux |
| `DPOTrainer` échoue | incompatibilité multimodale | Charger explicitement `Qwen3_5ForCausalLM` au lieu de la classe auto |
| VRAM > 14 Go à batch 1 | trop juste sur T4 | Réduire `max_seq_length`, ou passer en `gradient_checkpointing` |

**Noter la VRAM et le temps de génération** : ce sont les deux entrées de l'estimation compute,
qui est un livrable reporté de la semaine 5.

Une fois ces quatre points levés, la suite est dans [Week6_Checklist.md] du vault : juge,
mesure du juge, baseline B1, puis B3/B4 sur trois graines.